# Phase 4 — can retrieval score tell answerable from unanswerable?

Notebook 10 gated on `top similarity < 0.62` and refused **97% of unanswerable
questions — and 88% of answerable ones**. A 9-point gap is not detection.

Before designing a better guardrail, measure whether the signal carries any
information at all. This sweeps every threshold and computes the AUC, so the
answer is a number rather than an opinion.

**No LLM calls.** Retrieval only — about 4 minutes.

### 1. Setup

In [ ]:
# ---- Cell 1: setup (no LLM needed anywhere in this notebook) ----
import os, sys, shutil, subprocess, json, time, math, re
import numpy as np, statistics as st, torch
from google.colab import drive
print("CUDA:", torch.cuda.is_available())
if not os.path.isdir("/content/drive/MyDrive"): drive.mount("/content/drive")
ROOT="/content/drive/MyDrive"; P1FIX=f"{ROOT}/Phase1_Project/data_fix_output"
ROMA=f"{ROOT}/Phase2_Project/Roma_output"; OUT=f"{ROOT}/Phase4_Project"
os.makedirs(f"{OUT}/data", exist_ok=True)
PROJECT="/content/QuranicRAG"; os.makedirs(f"{PROJECT}/quranNLP/shared/data", exist_ok=True)
os.chdir(PROJECT)
shutil.rmtree("/content/_repo", ignore_errors=True)
subprocess.run(["git","clone","--depth","1",
 "https://github.com/Laiba-Noor/quranic-rag-hallucination-free.git","/content/_repo"],check=True)
CSV=f"{PROJECT}/quranNLP/shared/data/final_cross_reference_index.csv"
if not os.path.exists(CSV):
    shutil.copy(f"{P1FIX}/shared_data/final_cross_reference_index.csv", CSV)
try:
    import hnswlib, sentence_transformers, sklearn  # noqa
    print("deps present")
except ImportError:
    !pip install -q sentence-transformers hnswlib scikit-learn
print("SETUP OK")

### 2. Retrieval and questions

In [ ]:
# ---- Cell 2: retrieval + questions ----
import csv; csv.field_size_limit(sys.maxsize)
from sentence_transformers import SentenceTransformer
import hnswlib

rows=list(csv.DictReader(open(CSV, encoding="utf-8")))
VERSES=[(r["verse_key"], (r.get("clean_verse") or "").strip())
        for r in rows if (r.get("clean_verse") or "").strip()]
KEYS=[k for k,_ in VERSES]; VALID=set(KEYS)

model=SentenceTransformer("Omartificial-Intelligence-Space/GATE-AraBert-v1")
model.max_seq_length=64
emb=model.encode([t for _,t in VERSES], convert_to_numpy=True, normalize_embeddings=True,
                 show_progress_bar=True, batch_size=256).astype(np.float32)
ix=hnswlib.Index(space="cosine", dim=emb.shape[1])
ix.init_index(max_elements=len(VERSES), ef_construction=200, M=16)
ix.add_items(emb, ids=np.arange(len(VERSES))); ix.set_ef(128)

def sims(q, k=20):
    _,dist=ix.knn_query(model.encode([q], convert_to_numpy=True,
                                     normalize_embeddings=True), k=k)
    return 1.0 - dist[0]

def fetch(name,*c):
    dst=f"{OUT}/data/{name}"
    if os.path.exists(dst): return dst
    for p in c:
        if os.path.exists(p): shutil.copy(p,dst); return dst
    from google.colab import files
    print("Upload",name); up=files.upload(); shutil.copy(list(up.keys())[0],dst); return dst

aya=json.load(open(fetch("ayatec_records.json",
    "/content/_repo/Data/ayatec_records.json",
    f"{ROMA}/data/ayatec_records.json"), encoding="utf-8"))
ANS=[r["question"] for r in aya
     if r.get("question") and {v for v in r.get("verse_keys",[]) if v in VALID}]
UNANS=[r["question"] for r in aya
       if r.get("question") and r.get("question_type")=="zero_answer"]
print(f"answerable {len(ANS)} | unanswerable {len(UNANS)}")

### 3. Candidate signals

In [ ]:
# ---- Cell 3: candidate answerability signals ----
def features(q):
    s=sims(q, 20)
    return {
      "top1":      float(s[0]),
      "top5_mean": float(s[:5].mean()),
      "margin":    float(s[0]-s[4]),          # is the top result distinctive?
      "n_above_55":float((s>=0.55).sum()),
      "std_top10": float(s[:10].std()),
    }

t=time.time()
FA=[features(q) for q in ANS]
FU=[features(q) for q in UNANS]
print(f"scored {len(FA)+len(FU)} questions in {time.time()-t:.0f}s")

NAMES=list(FA[0])
print(f"\n{'signal':<12}{'answerable':>14}{'unanswerable':>15}{'gap':>9}")
print("-"*50)
for n in NAMES:
    a=st.mean(f[n] for f in FA); u=st.mean(f[n] for f in FU)
    print(f"{n:<12}{a:>14.4f}{u:>15.4f}{a-u:>+9.4f}")

### 4. Separation

In [ ]:
# ---- Cell 4: can any of them actually separate the two? ----
from sklearn.metrics import roc_auc_score

y=[1]*len(FA)+[0]*len(FU)          # 1 = answerable
print(f"{'signal':<12}{'AUC':>8}   interpretation")
print("-"*54)
best=None
for n in NAMES:
    x=[f[n] for f in FA]+[f[n] for f in FU]
    auc=roc_auc_score(y,x)
    verdict=("useless (chance)" if auc<0.60 else
             "weak"            if auc<0.70 else
             "moderate"        if auc<0.80 else "usable")
    print(f"{n:<12}{auc:>8.3f}   {verdict}")
    if best is None or auc>best[1]: best=(n,auc)
print(f"\nbest signal: {best[0]}  AUC={best[1]:.3f}")
print("AUC 0.5 = coin flip. A guardrail needs >=0.8 to be worth shipping.")

### 5. Threshold sweep

In [ ]:
# ---- Cell 5: the trade-off at every threshold ----
sig=best[0]
xa=sorted(f[sig] for f in FA); xu=sorted(f[sig] for f in FU)
lo,hi=min(xa+xu), max(xa+xu)

print(f"sweeping {sig}\n")
print(f"{'thresh':>8}{'refuse unans':>15}{'refuse ans':>13}{'gap':>9}{'youden':>9}")
print("-"*56)
rowsout=[]
for th in [lo+(hi-lo)*i/24 for i in range(25)]:
    ru=sum(1 for v in xu if v<th)/len(xu)      # correct refusals
    ra=sum(1 for v in xa if v<th)/len(xa)      # over-refusals
    rowsout.append((th,ru,ra,ru-ra))
    print(f"{th:>8.3f}{ru:>15.3f}{ra:>13.3f}{ru-ra:>+9.3f}{ru-ra:>9.3f}")

th,ru,ra,gap=max(rowsout, key=lambda r:r[3])
print(f"\nBEST OPERATING POINT  threshold={th:.3f}")
print(f"  refuses {ru:.1%} of unanswerable, {ra:.1%} of answerable  (gap {gap:+.1%})")
print(f"\nfor comparison, the 0.62 gate used in notebook 10:")
i=min(range(len(rowsout)), key=lambda j: abs(rowsout[j][0]-0.62))
print(f"  threshold={rowsout[i][0]:.3f} -> {rowsout[i][1]:.1%} / {rowsout[i][2]:.1%}"
      f"  (gap {rowsout[i][3]:+.1%})")

json.dump({"auc":{n: float(roc_auc_score(y,[f[n] for f in FA]+[f[n] for f in FU]))
                  for n in NAMES},
           "best_signal":sig, "best_threshold":th,
           "refuse_unanswerable":ru, "refuse_answerable":ra},
          open(f"{OUT}/phase4_answerability.json","w"), indent=2)
print("\nsaved:", f"{OUT}/phase4_answerability.json")

### 6. Verdict

In [ ]:
# ---- Cell 6: what this means ----
print("If the best AUC is below ~0.7, retrieval score cannot carry the guardrail.")
print("The system would need a different answerability signal, for example:")
print("  - asking the LLM to judge sufficiency in a separate call")
print("  - a trained classifier over the retrieved passages")
print("  - agreement between two retrievers")
print()
print(f"measured best: {best[0]} at AUC {best[1]:.3f}")
print(f"ceiling for a threshold rule: {gap:+.1%} separation "
      f"({ru:.0%} of unanswerable refused at {ra:.0%} over-refusal)")